In [1]:
import ipyparallel as ipp
n = 10
cluster = ipp.Cluster(engines = "mpi", n = n)
rc = cluster.start_and_connect_sync()
rc.activate()
view = rc[:]

Starting 10 engines with <class 'ipyparallel.cluster.launcher.MPIEngineSetLauncher'>


  0%|          | 0/10 [00:00<?, ?engine/s]

In [2]:
%%px 

import logging
from pathlib import Path

In [3]:
%%px

def write_partitioned_mesh(filename: Path):
    import subprocess

    from mpi4py import MPI

    import dolfinx

    import adios4dolfinx

    # Create a simple unit square mesh
    mesh = dolfinx.mesh.create_box(
        MPI.COMM_WORLD,
        [[0.0, 0.0, 0.0], [0.1, 0.1, 0.2]],
        [40, 40, 80],
        cell_type=dolfinx.mesh.CellType.hexahedron,
        ghost_mode=dolfinx.mesh.GhostMode.shared_facet,
    )

    # Write mesh checkpoint
    adios4dolfinx.write_mesh(filename, mesh, engine="BP4", store_partition_info=True)
    # Inspect checkpoint on rank 0 with `bpls`
    if mesh.comm.rank == 0:
        output = subprocess.run(["bpls", "-a", "-l", filename], capture_output=True)
        print(output.stdout.decode("utf-8"))



In [4]:
%%px 

import gmsh
import os
import time
import numpy as np
from dolfinx.io import gmshio
from mpi4py import MPI
import pyvista as pv
from dolfinx.plot import vtk_mesh

def generate_layered_meshes(layer_height, part_height, lx=1.0, ly=1.0, output_folder="layers"):
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 1)

    os.makedirs(output_folder, exist_ok=True)
    n_layers = int(part_height / layer_height)
    

    for i in range(42, 43):
        gmsh.model.add(f"layer_{i}")

        fine_thickness = 3 * layer_height # top 3 layers are finer
        fine_start_z = max(0, part_height - fine_thickness)
        max_size = 10 * layer_height
        min_size = layer_height / 2

        box_id = gmsh.model.mesh.field.add("Box")
        gmsh.model.mesh.field.setNumber(box_id, "VIn", min_size)
        gmsh.model.mesh.field.setNumber(box_id, "VOut", max_size)
        gmsh.model.mesh.field.setNumber(box_id, "XMin", 0)
        gmsh.model.mesh.field.setNumber(box_id, "XMax", lx)
        gmsh.model.mesh.field.setNumber(box_id, "YMin", 0)
        gmsh.model.mesh.field.setNumber(box_id, "YMax", ly)
        gmsh.model.mesh.field.setNumber(box_id, "ZMin", part_height - 3 * layer_height)
        gmsh.model.mesh.field.setNumber(box_id, "ZMax", part_height)
        gmsh.model.mesh.field.setNumber(box_id, "Thickness", 5 * layer_height) # transition thickness 

        gmsh.model.mesh.field.setAsBackgroundMesh(box_id)

        box = gmsh.model.occ.addBox(0, 0, 0, lx, ly, part_height)
        
        # Create a box (hexahedral mesh)
        gmsh.model.occ.synchronize()
        gmsh.model.addPhysicalGroup(3, [box], 1)
        gmsh.model.setPhysicalName(3, 1, f"volume_{i}")

        surfaces = gmsh.model.getBoundary([(3, box)], oriented=False, recursive=False)
        top, bottom, sides = [], [], []

        for dim, tag in surfaces:
            com_x, com_y, com_z = gmsh.model.occ.getCenterOfMass(dim, tag)
            if abs(com_z - part_height) < 1e-9:
                top.append(tag)
            elif abs(com_z) < 1e-6:
                bottom.append(tag)
            else:
                sides.append(tag)

        gmsh.model.addPhysicalGroup(2, top, tag=13)
        gmsh.model.setPhysicalName(2, 13, "top")

        gmsh.model.addPhysicalGroup(2, bottom, tag=14)
        gmsh.model.setPhysicalName(2, 14, "bottom")

        gmsh.model.addPhysicalGroup(2, sides, tag=15)
        gmsh.model.setPhysicalName(2, 15, "sides")


        # Structured mesh
        gmsh.model.mesh.generate(3)
        gmsh.model.mesh.removeDuplicateNodes()

        #gmsh.model.mesh.optimize("Netgen")

        path = os.path.join(output_folder, f"layer_{i}.msh")
        gmsh.write(path)
        gmsh.model.remove()

    gmsh.finalize()



def load_first_layer(msh_file):
    # Convert mesh if needed and import
    print(type(msh_file))
    mesh_, cell_tags, facet_tags = gmshio.read_from_msh(msh_file, comm = MPI.COMM_WORLD, rank=0, gdim=3)
    return mesh_, cell_tags, facet_tags


def plot_dolfinx_mesh(mesh_, cell_type=3):
    """
    Plot a dolfinx mesh using PyVista.
    
    Parameters:
    - mesh: dolfinx.cpp.mesh.Mesh
    - cell_type: 2 for triangle (2D), 3 for tetrahedron (3D)
    """
    grid = pv.UnstructuredGrid(*vtk_mesh(mesh_, mesh_.topology.dim))

    plotter = pv.Plotter()
    plotter.add_mesh(grid, show_edges=True, color="lightblue", opacity=0.7)
    plotter.show()


# ---- Run it ----
if __name__ == "__main__":
    layer_height = 1000 * 1e-6
    part_height = 0.1 
    #print("This many files will be generated: ", int(part_height / layer_height))

    #if MPI.COMM_WORLD.rank == 0:
    #    generate_layered_meshes(layer_height, part_height, lx = 0.1, ly = 0.1, output_folder="layers")
    #print(f"Generated mesh file: {msh_file}")
    #start_time = time.time()
    #mesh1, cell_tags1, facet_tags1 = load_first_layer(os.path.join("layers", "layer_42.msh"))
    #end_time = time.time()
    #print(f"Time taken to load mesh: {(end_time - start_time)*1000} ms")

    #print(f"Loaded mesh with {mesh1.topology.index_map(3).size_local} cells.")

    #plot_dolfinx_mesh(mesh1)

#mesh0, cell_tags0, facet_tags0 = load_first_layer(os.path.join("layers", "layer_42.msh"))
start_time = time.time()
mesh0, cell_tags0, facet_tags0 = load_first_layer(os.path.join("layers", "layer_42.msh")) # /mnt/ramdisk/ slower than layers
end_time = time.time()
print(f"Time taken to load mesh: {(end_time - start_time)*1000} ms")
#next up: https://jsdokken.com/adios4dolfinx/docs/partitioned_mesh.html

[stdout:2] <class 'str'>
Time taken to load mesh: 6712.8705978393555 ms


[stdout:4] <class 'str'>
Time taken to load mesh: 6745.623588562012 ms


[stdout:9] <class 'str'>
Time taken to load mesh: 6729.196071624756 ms


[stdout:7] <class 'str'>
Time taken to load mesh: 6732.192039489746 ms


[stdout:5] <class 'str'>
Time taken to load mesh: 6735.079526901245 ms


...

[stdout:8] <class 'str'>
Time taken to load mesh: 6693.696022033691 ms


[stdout:6] <class 'str'>
Time taken to load mesh: 6709.644079208374 ms


[stdout:1] <class 'str'>
Time taken to load mesh: 6717.49210357666 ms


[stdout:0] <class 'str'>
Info    : Reading 'layers/layer_42.msh'...
Info    : 27 entities
Info    : 234666 nodes
Info    : 1401499 elements
Info    : Done reading 'layers/layer_42.msh'
Time taken to load mesh: 6722.02205657959 ms


%px:   0%|          | 0/10 [00:00<?, ?tasks/s]

In [5]:
%%px 

if MPI.COMM_WORLD.rank == 0:
    plot_dolfinx_mesh(mesh1)

[0:execute]
---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[4], line 2
      1 if MPI.COMM_WORLD.rank == 0:
----> 2     plot_dolfinx_mesh(mesh1)

NameError: name 'mesh1' is not defined


AlreadyDisplayedError: 1 errors

In [6]:
%%px 

def write_partitioned_mesh_XDMF(filename: Path):
    import dolfinx
    from dolfinx.io import XDMFFile
    mesh, cell_tags, facet_tags = load_first_layer(os.path.join("layers", "layer_42.msh"))
    cell_tags.name = "cell_tags" 
    facet_tags.name = "facet_tags" # default was "Facet tags"
    
    file = dolfinx.io.XDMFFile(mesh.comm, filename, "w") # file_mode: https://www.geeksforgeeks.org/file-mode-in-python/; "w" stands for write
    file.write_mesh(mesh)
    file.write_meshtags(facet_tags, mesh.geometry)
    file.write_meshtags(cell_tags, mesh.geometry)


def read_partitioned_mesh_XDMF(filename: Path):
    import dolfinx
    from dolfinx.io import XDMFFile

    file = dolfinx.io.XDMFFile(MPI.COMM_WORLD, filename, "r")
    mesh = file.read_mesh()
    tdim = mesh.topology.dim
    mesh.topology.create_connectivity(tdim - 1, tdim)
    #mesh.topology.create_connectivity(1, tdim)
    cell_tags = file.read_meshtags(mesh, "cell_tags")
    facet_tags = file.read_meshtags(mesh, "facet_tags")
    
    return mesh, cell_tags, facet_tags

write_partitioned_mesh_XDMF("test.xdmf")

start_time = time.time()
mesh1, cell_tags1, facet_tags1 = read_partitioned_mesh_XDMF("test.xdmf")
end_time = time.time()
print(f"Time taken to load mesh: {(end_time - start_time)*1000} ms")

[stdout:3] <class 'str'>
Time taken to load mesh: 1709.6397876739502 ms


[stdout:5] <class 'str'>
Time taken to load mesh: 1709.6517086029053 ms


[stdout:1] <class 'str'>
Time taken to load mesh: 1709.6307277679443 ms


[stdout:0] <class 'str'>
Info    : Reading 'layers/layer_42.msh'...
Info    : 27 entities
Info    : 234666 nodes
Info    : 1401499 elements
Info    : Done reading 'layers/layer_42.msh'
Time taken to load mesh: 1709.6400260925293 ms


[stdout:2] <class 'str'>
Time taken to load mesh: 1709.611415863037 ms


...

[stdout:7] <class 'str'>
Time taken to load mesh: 1709.6858024597168 ms


[stdout:6] <class 'str'>
Time taken to load mesh: 1709.641456604004 ms


[stdout:9] <class 'str'>
Time taken to load mesh: 1709.6774578094482 ms


[stdout:8] <class 'str'>
Time taken to load mesh: 1709.6600532531738 ms


%px:   0%|          | 0/10 [00:00<?, ?tasks/s]

In [7]:
%%px

def write_partitioned_mesh(filename: Path):
    import subprocess
    from mpi4py import MPI
    import dolfinx
    import adios4dolfinx

    mesh, cell_tags, facet_tags = load_first_layer(os.path.join("layers", "layer_42.msh"))
    
    # Write mesh checkpoint
    adios4dolfinx.write_mesh(filename, mesh, engine="BP4", store_partition_info=True)
    adios4dolfinx.write_meshtags(filename, mesh, cell_tags, engine="BP4", meshtag_name = "cells")
    adios4dolfinx.write_meshtags(filename, mesh, facet_tags, engine="BP4", meshtag_name = "facets")
    # Inspect checkpoint on rank 0 with `bpls`
    if mesh.comm.rank == 0:
        output = subprocess.run(["bpls", "-a", "-l", filename], capture_output=True)
        print(output.stdout.decode("utf-8"))


def read_partitioned_mesh(filename: Path, read_from_partition: bool = True):
    from mpi4py import MPI

    import adios4dolfinx

    prefix = f"{MPI.COMM_WORLD.rank + 1}/{MPI.COMM_WORLD.size}: "
    try:
        mesh = adios4dolfinx.read_mesh(
            filename, comm=MPI.COMM_WORLD, engine="BP4", read_from_partition=read_from_partition
        )
        cell_tags = adios4dolfinx.read_meshtags(filename, mesh, meshtag_name = "cells", engine="BP4")
        facet_tags = adios4dolfinx.read_meshtags(filename, mesh, meshtag_name = "facets", engine="BP4")

        tdim = mesh.topology.dim
        mesh.topology.create_connectivity(tdim - 1, tdim)

        print(f"{prefix} Mesh: {mesh.name} read successfully with {read_from_partition=}")
    except ValueError as e:
        print(f"{prefix} Caught exception: ", e)

    return mesh, cell_tags, facet_tags

In [8]:
%%px 

import time 

mesh_file = Path("partitioned_mesh.bp")
#write_partitioned_mesh(mesh_file)

start_time = time.time()
mesh2, cell_tags2, facet_tags2 = read_partitioned_mesh(mesh_file, True)
end_time = time.time()
print(f"Time taken to read mesh: {(end_time - start_time)*1000} ms")
print(f"Loaded mesh with {mesh1.topology.index_map(3).size_local} cells.")
#print(len(mesh.cell_tags.values), "DOFs for this process")

[stdout:0] 1/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1201.0531425476074 ms
Loaded mesh with 128724 cells.


[stdout:4] 5/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1219.501256942749 ms
Loaded mesh with 129263 cells.


[stdout:6] 7/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1223.1879234313965 ms
Loaded mesh with 129157 cells.


[stdout:2] 3/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1224.984884262085 ms
Loaded mesh with 129647 cells.


[stdout:7] 8/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1225.876808166504 ms
Loaded mesh with 129193 cells.


[stdout:1] 2/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1231.5773963928223 ms
Loaded mesh with 129920 cells.


[stdout:9] 10/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1237.7893924713135 ms
Loaded mesh with 129172 cells.


[stdout:3] 4/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1246.0906505584717 ms
Loaded mesh with 129006 cells.


[stdout:8] 9/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1245.710849761963 ms
Loaded mesh with 129190 cells.


[stdout:5] 6/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1250.774621963501 ms
Loaded mesh with 129159 cells.


In [ ]:
%%px 

import cProfile, pstats
profiler = cProfile.Profile()
profiler.enable()
_, _, _ = _(os.path.join("layers", "layer_42.msh")) # /mnt/ramdisk/ slower than layers
profiler.disable()
stats = pstats.Stats(profiler).sort_stats('ncalls')
stats.print_stats()

[stdout:0] <class 'str'>
Info    : Reading 'layers/layer_42.msh'...
Info    : 27 entities
Info    : 234666 nodes
Info    : 1401499 elements
Info    : Done reading 'layers/layer_42.msh'
         10781 function calls (10046 primitive calls) in 6.488 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
1105/1077    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
      894    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      639    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
   412/68    0.000    0.000    0.005    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      322    0.000    0.000    0.000

[stdout:1] <class 'str'>
         6691 function calls (6024 primitive calls) in 6.452 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      753    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      604    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
  500/492    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
   412/68    0.000    0.000    0.004    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      306    0.000    0.000    0.000    0.000 {built-in method _abc._abc_subclasscheck}
      306    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
      303    0.000    0.000 

[stdout:5] <class 'str'>
         6827 function calls (6178 primitive calls) in 6.416 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      755    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      605    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
  543/535    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
   412/68    0.000    0.000    0.003    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      306    0.000    0.000    0.000    0.000 {built-in method _abc._abc_subclasscheck}
      306    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
      304    0.000    0.000 

[stdout:6] <class 'str'>
         6875 function calls (6210 primitive calls) in 6.429 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      762    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      605    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
  544/536    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
   412/68    0.000    0.000    0.002    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      306    0.000    0.000    0.000    0.000 {built-in method _abc._abc_subclasscheck}
      306    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
      304    0.000    0.000 

[stdout:7] <class 'str'>
         6875 function calls (6210 primitive calls) in 6.453 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      762    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      605    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
  544/536    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
   412/68    0.000    0.000    0.002    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      306    0.000    0.000    0.000    0.000 {built-in method _abc._abc_subclasscheck}
      306    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
      304    0.000    0.000 

...

Out[4:10]: <pstats.Stats at 0x7f802f207d90>

Out[2:10]: <pstats.Stats at 0x7f6d5129fd90>

Out[7:10]: <pstats.Stats at 0x7f2458667d90>

Out[3:10]: <pstats.Stats at 0x7fe49cb2bd90>

Out[0:10]: <pstats.Stats at 0x7fc7db613d90>

In [9]:
%%px 

local_count0 = (facet_tags0.values == 13).sum()
local_count0 += (facet_tags0.values == 14).sum()
local_count0 += (facet_tags0.values == 15).sum()

local_count1 = (facet_tags1.values == 13).sum()
local_count1 += (facet_tags1.values == 14).sum()
local_count1 += (facet_tags1.values == 15).sum()

local_count2 = (facet_tags2.values == 13).sum()
local_count2 += (facet_tags2.values == 14).sum()
local_count2 += (facet_tags2.values == 15).sum()

print(local_count0)
print(local_count1)
print(local_count2)


[stdout:0] 12112
12245
12112


[stdout:1] 10159
10408
10159


[stdout:4] 11943
12061
11943


[stdout:3] 9619
9773
9619


[stdout:2] 10576
11432
10576


[stdout:9] 10964
10877
10964


[stdout:6] 12163
12281
12163


[stdout:5] 10315
10290
10315


[stdout:8] 9064
9213
9064


[stdout:7] 12153
12262
12153


In [104]:
total_count0 = sum(rc[:]['local_count0'])
total_count1 = sum(rc[:]['local_count1'])
total_count2 = sum(rc[:]['local_count2'])

print("Total facets with tag:", total_count0)
print("Total facets with tag:", total_count1)
print("Total facets with tag:", total_count2)
print(total_count1-total_count2)

Total facets with tag: 109068
Total facets with tag: 110842
Total facets with tag: 109068
1774


In [69]:
%%px 

mesh1, cell_tags1, facet_tags1 = load_first_layer(os.path.join("layers", "layer_66.msh"))

print(f"Loaded mesh with {mesh1.topology.index_map(3).size_local} cells.")

#plot_dolfinx_mesh(mesh1)


[stdout:0] <class 'str'>
Info    : Reading 'layers/layer_66.msh'...
Info    : 27 entities
Info    : 183760 nodes
Info    : 1122936 elements
Info    : Done reading 'layers/layer_66.msh'
Loaded mesh with 373908 cells.


[stdout:1] <class 'str'>
Loaded mesh with 373778 cells.


[stdout:2] <class 'str'>
Loaded mesh with 373784 cells.


%px:   0%|          | 0/3 [00:00<?, ?tasks/s]